# VDocRAG demo (Path B) -- run in Colab
ColQwen2.5-3B (retrieval) + Qwen2.5-VL-7B-Instruct (generation), served via Streamlit + localtunnel.

In [ ]:
# 1. Mount Drive -- only for the small, worth-persisting stuff (your indexed
# PDFs' page images + embeddings). The multi-GB model weight cache does NOT
# go on Drive: writing ~7-15GB of weights over Drive's FUSE mount is slow
# enough to look like a hang, and it eats into Drive quota for no real
# benefit -- redownloading from HF's CDN to local disk each fresh session is
# both faster and free (space-wise). HF_HOME instead lives on local Colab
# disk: fast, but ephemeral (redownloads on every fresh runtime).
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/vdocrag-pathb'  # small: index + page images only
os.makedirs(PROJECT_DIR, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_cache'  # local disk, NOT Drive -- large, ephemeral
os.environ['VDOCRAG_DATA_DIR'] = f'{PROJECT_DIR}/data'  # Drive -- small, persisted


In [ ]:
# 1.5. Sanity check: fail fast here, not five cells and 90 seconds later,
# if this session has no GPU attached. If this errors, go to:
# Runtime > Change runtime type > Hardware accelerator > select a GPU
# (e.g. T4) > Save, then Runtime > Restart session, then re-run from the top.
!nvidia-smi || (echo '\n>>> No GPU visible. Runtime > Change runtime type > select a GPU, then Restart session.' && exit 1)

In [ ]:
# 2. System deps for PDF rasterization
!apt-get -qq update && apt-get -qq install -y poppler-utils

In [ ]:
# 3. Get the code -- first time: git clone; after that: git pull
# Folder name is derived from REPO_URL so it can never drift out of sync
# with the repo (previously hardcoded to a stale 'vdocrag-pathb' placeholder
# that didn't match the actual repo name -- caused clone/cd/pull/install to
# all silently no-op or fail downstream).
REPO_URL = 'https://github.com/thejainamjain/vdocrag-implementation.git'  # <-- set this
%cd /content
import os
repo_name = REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
if not os.path.exists(repo_name):
    !git clone {REPO_URL}
%cd {repo_name}
!git pull

In [ ]:
# 4. Install Python deps
!pip install -q -r requirements-colab.txt

In [ ]:
# 5. Launch Streamlit in the background, then tunnel it out with cloudflared.
# Swapped from localtunnel: localtunnel's free tier is unreliable at serving
# every static asset request under a Streamlit app's many JS chunks, which
# surfaces in the browser as `TypeError: Importing a module script failed.`
# (a chunk fetch silently failing/getting mangled) even when the Streamlit
# backend itself is running fine. cloudflared's quick tunnels are free, need
# no signup, skip localtunnel's password-page step entirely, and are
# materially more reliable for this. If you still see module-import errors
# after this swap, it's very likely a browser extension (ad blocker /
# tracking protection) blocking the requests -- test in an incognito window
# with extensions off.
#
# Kill anything left over from a previous run of this cell first.
import subprocess, time, re

!pkill -f 'streamlit run' 2>/dev/null
!pkill -f 'cloudflared' 2>/dev/null
!pkill -f 'localtunnel' 2>/dev/null
!fuser -k 8501/tcp 2>/dev/null
time.sleep(2)

subprocess.Popen([
    'streamlit', 'run', 'app.py',
    '--server.port', '8501', '--server.headless', 'true',
])
time.sleep(8)

if not __import__('os').path.exists('cloudflared'):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print()
if url:
    print(f'Your app is live at: {url}')
    print('(open in a NEW tab / incognito rather than reusing an old tab)')
else:
    print('Timed out waiting for cloudflared to print a URL -- check the cell '
          'output above for errors.')